## General TO DOs:
Most articles don't get locations from NER Pass, so check if it's even better than the LLM Pass or if it's worth the extra time (roughly double):
* Benchmark LLM Pass vs NER (Time and Accuracy). Compare FAC/ORG/LOC given, how good they are, and time taken.
* Check if NER is necessary on LLM Pass.

Improve LLM:
* Slice texts so they fit LLM's token limit of 2048.
* Add better prompt that allows it to not force a boston location
* Make it give a formatted response so NER isn't necesssary.
* Mentions that it is in boston even if it isn't.
* If it says it isn't in Boston, NER still grabs a location. It shouldn't.

Others:
* Populate list of unwanted locations

* ### Still need to Review Topic Modeling


In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# TODO: Add better prompt that allows it to not force a boston location
# TODO: Make it give a formatted response so NER isn't necesssary.

# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

In [7]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [8]:
import spacy
from span_marker import SpanMarkerModel

In [9]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [10]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [11]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [12]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [13]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [14]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [15]:
import time

def check_time(func):
    def sec_to_hms(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        remaining_seconds = round(seconds % 60)
        return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [16]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [17]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(5)
# raw_df = full_df
len(raw_df)


5

In [18]:
# raw_df = pd.read_csv(sample_data_path)

In [19]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Article,"Due to COVID, JFK Library celebrates President...","Due to COVID, JFK Library celebrates President...",Kirk Carapezza,NaN,Local News,NaN,/local-news/2022/02/20/due-to-covid-jfk-librar...,Sun Feb 20 11:49:12 EST 2022,TRUE,The John F. Kennedy Presidential Library and M...
8714,00000180-895d-d69c-ade1-b95d5f6c0001,Article,4 things to know as the Fed embarks on its big...,4 things to know as the Fed embarks on its big...,Scott Horsley,NaN,National News,NaN,/national-news/2022/05/03/4-things-to-know-as-...,Tue May 03 05:00:00 EDT 2022,TRUE,The Federal Reserve is about to deliver its bi...
12349,00000186-8013-d717-adce-8a1f9b140002,Article,He watched the Koons 'balloon dog' fall and sh...,He watched the Koons 'balloon dog' fall and sh...,Lauren Hodges,NaN,National News,NaN,/national-news/2023/02/23/he-watched-the-koons...,Thu Feb 23 15:49:00 EST 2023,TRUE,Welcome to a new NPR series where we spotlight...
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Article,Online Offers To Escort Seniors To Vaccine App...,Online Offers To Escort Seniors To Vaccine App...,Haley Lerner,NaN,Local News,NaN,/local-news/2021/02/12/online-offers-to-escort...,Fri Feb 12 20:17:42 EST 2021,TRUE,People are posting online solicitations to giv...
8377,00000180-0377-d400-a7d5-17ffa6080000,Article,Putin's daughters were just sanctioned. Here's...,Putin's daughters were just sanctioned. Here's...,Laurel Wamsley,NaN,International News,NaN,/international-news/2022/04/07/putins-daughter...,Thu Apr 07 05:00:00 EDT 2022,TRUE,The U.S. has announced additional sanctions ag...


The ML Model honestly just needs the `id`, `header`, and `body`.

In [20]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [21]:
# For Testing Purposes Only
# df = df[:20]

Remove Duplicates (if any)

In [22]:
duplicates = df.duplicated(subset=['hl1'])

In [23]:
print(duplicates.value_counts())

False    5
Name: count, dtype: int64


In [24]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [25]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

100%|██████████| 5/5 [00:00<?, ?it/s]


Clean the Body and Header with Regex

In [26]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 5/5 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [27]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [28]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                return location
    return None

In [29]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 5/5 [00:00<00:00, 2494.83it/s]


In [30]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
JFK Library    1
BU             1
Name: count, dtype: int64

### NER Code First Pass

In [31]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, firstPass):
    print(entities)

    if (firstPass): 
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
        else:
            return None
    
    # Process for the LLM Prediction Pass
    else:
        first_org = None
        for entity in entities:
            # If it's a valid facility, return it
            if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
                return entity.text
            
            # If it's a valid organization, save it (but don't return in case there's a facility later on)
            if (first_org == None and entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
                first_org = entity.text
        else:             
            return first_org # Return regardless of whether it's None or not 
        

In [32]:
# Run NER on the body of the article and return first valid facility
def run_NER(text, firstPass=True):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, firstPass)
        
    except Exception as error:
        print(error)
        return None

In [33]:
# Old way of running the NER
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return run_NER(article['body'])
    except Exception as error:
        print(error)
        return None

# df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

In [34]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    # Process each chunk and return if a valid facilty is found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             return result
    return None

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [35]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have an explicit location in the title
    if (article['Explicit_Pass'] != None): 
        print(f"Has location from title: {article['hl1']}")
        return None

    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text)
    except Exception as error:
        print(error)

In [36]:
df["NER_Pass"] = df.progress_apply(handle_chunk_processing, axis=1)

  0%|          | 0/5 [00:00<?, ?it/s]

Has location from title: Due to COVID JFK Library celebrates Presidents Day online
Time taken: 00:00:00
(The Federal Reserve, two day, Tuesday, half, more than two decades, Fed, 40 years, Fed, the months to come, Fed)
(Fed, the Commerce Department, last week, 6.6%, the 12 months ending in March, more than three, Fed, Fed)
(Fed, quarter, March, this week, first, half, Fed, Fed, Fed, March, nearly full percentage points, this year)
(next year, Fed, Jerome Powell, Powell, Fed, quarter, Powell, International Monetary Fund, last month)
(5%, Fed, less than 3%, year ago, about $370, monthly, Fed, Powell)
(Fed, Deutsche Bank, German, last week, next year, last week, Fed, Fed)


 60%|██████    | 3/5 [02:31<01:40, 50.45s/it]

(Fed, Kathy Bostjancic, Oxford Economics, 2022, NPR)
Time taken: 00:02:31
Has location from title: He watched the Koons balloon dog fall and shatter...and wants to buy the remains
Time taken: 00:00:00
(Massachusetts, Charlie Baker, Wednesday, Massachusetts, age 75, 19, Baker, Thursday)
(Baker, Twitter, Eileen Cotter Wright, over 75, boston, Massachusetts, 35 year old, Cotter Wright, Quincy)
(Gillette,)
(Cotter Wright, Craigslist, dozens, One, Craigslist, two, Northampton, 75)
(n95, Boston, 275, 27yr old, Greenfield, two, 100, 200)
(one, 30 year old, MA, 75, 19, one)


100%|██████████| 5/5 [04:51<00:00, 60.04s/it]

(75, COVID 19, 19, Haley Lerner, GBH News Beat the Press)
Time taken: 00:02:20
(U.S., Russia, Russian, Ukraine, two, Vladimir Putin, Putin, Biden, Wednesday, Putin, the Treasury Department, Maria Vladimirovna Vorontsova, Katerina Vladimirovna Tikhonova, One)
(Putin, Putin, Putin, Lyudmila Putina, Aeroflot, Putin, 1983, two, three decades, One, Putin, 2015, BBC, Russia, Russia, Putin)
(three, Russian, Putin, years, the last several years, Putin, Maria Vladimirovna Vorontsova, Faassen)
(Maria Putina, Masha, 1985, Vorontsova, the Endocrinology Research Center Moscow, Vorontsova, the Russian Society of Young Endocrinologists, English, German, French, Dutch, Vorontsova, billions of dollars, Kremlin, Putin, the Treasury Department, 2019, Russian)
(BBC Russia, Vorontsova, Russian, Vorontsova, Dutch, Jorrit Joost Faassen, BBC, Russian, Gazprom, two, second, Katerina Vladimirovna Tikhonova, Germany, 1986, Putin, KGB, Katya, Tikhonova)
(Reuters, years, Ivan Klimov, Today, the Government of Russi

100%|██████████| 5/5 [08:05<00:00, 97.11s/it]

(Olympic, Alina Kabaeva, Putin, Putin, Kabaeva, Newsweek, Putin, two, Kabaeva, Switzerland, 2022, NPR)
Time taken: 00:03:14


In [37]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None


### Llama Prediction

In [38]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet. Then run NER on the LLM prediction
@check_time
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            llama_prediction = run_llm(article['hl1'], article['body'])
            print(llama_prediction)
            return run_NER(llama_prediction, True)
    except Exception as error:
        print(error)
        return None

In [39]:
df['NER_Prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/5 [00:00<?, ?it/s]

Has location from title or NER: Due to COVID JFK Library celebrates Presidents Day online
Time taken: 00:00:00
  1. Y - The article is talking about a region of Boston, specifically the Federal Reserve's meeting location in Boston.
2. The specific location within Boston that the article mentions is the Federal Reserve Bank of Boston, which is located at 600 Atlantic Avenue, Boston, MA 02110.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision include:
* The Federal Reserve Bank of Boston
* The Commerce Department
* Global supply chains
* Employers
* Oxford Economics

Based on the information provided in the article, it appears that the Federal Reserve is set to raise interest rates at its meeting in Boston this week, with the goal of curbing inflation. The article highlights the urgency with which the Fed is approaching inflation, as prices continue to climb at the fastest pace in 40 years. The Fed is expected to keep push


llama_print_timings:        load time =   95946.74 ms
llama_print_timings:      sample time =      90.60 ms /   256 runs   (    0.35 ms per token,  2825.73 tokens per second)
llama_print_timings: prompt eval time =  104839.58 ms /  1121 tokens (   93.52 ms per token,    10.69 tokens per second)
llama_print_timings:        eval time =   47381.54 ms /   255 runs   (  185.81 ms per token,     5.38 tokens per second)
llama_print_timings:       total time =  152972.46 ms /  1376 tokens


  1. Y - The article is talking about a region of Boston, specifically the Federal Reserve's meeting location in Boston.
2. The specific location within Boston that the article mentions is the Federal Reserve Bank of Boston, which is located at 600 Atlantic Avenue, Boston, MA 02110.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision include:
* The Federal Reserve Bank of Boston
* The Commerce Department
* Global supply chains
* Employers
* Oxford Economics

Based on the information provided in the article, it appears that the Federal Reserve is set to raise interest rates at its meeting in Boston this week, with the goal of curbing inflation. The article highlights the urgency with which the Fed is approaching inflation, as prices continue to climb at the fastest pace in 40 years. The Fed is expected to keep pushing borrowing costs higher in the months to come, and is also expected to announce plans to gradually reduce th

 60%|██████    | 3/5 [03:21<02:14, 67.32s/it]

...

  Based on the article provided, here is my response:
1. Y - The article is talking about specific locations within Boston, Massachusetts, as it mentions the city's Endocrinology Research Center and Moscow State University.
2. The specific location within Boston mentioned in the article is the Endocrinology Research Center, where Maria Vladimirovna Vorontsova works as a pediatric endocrinologist and genetics researcher. The center is located at an undisclosed address in Boston.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Endocrinology Research Center, Moscow (where Maria Vladimirovna Vorontsova works)
* Moscow State University (where Katerina Vladimirovna Tikhonova is a member of the economic board and heads the National Intellectual Reserve Center)
* St. Petersburg (mentioned as the hometown of longtime acquaintance of Putin, Kirill Shamalov)
* Biarritz, France (mentioned as the location of a villa owned by


llama_print_timings:        load time =   95946.74 ms
llama_print_timings:      sample time =     100.62 ms /   256 runs   (    0.39 ms per token,  2544.20 tokens per second)
llama_print_timings: prompt eval time =  118955.53 ms /  1342 tokens (   88.64 ms per token,    11.28 tokens per second)
llama_print_timings:        eval time =   50935.31 ms /   255 runs   (  199.75 ms per token,     5.01 tokens per second)
llama_print_timings:       total time =  170673.60 ms /  1597 tokens


  Based on the article provided, here is my response:
1. Y - The article is talking about specific locations within Boston, Massachusetts, as it mentions the city's Endocrinology Research Center and Moscow State University.
2. The specific location within Boston mentioned in the article is the Endocrinology Research Center, where Maria Vladimirovna Vorontsova works as a pediatric endocrinologist and genetics researcher. The center is located at an undisclosed address in Boston.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Endocrinology Research Center, Moscow (where Maria Vladimirovna Vorontsova works)
* Moscow State University (where Katerina Vladimirovna Tikhonova is a member of the economic board and heads the National Intellectual Reserve Center)
* St. Petersburg (mentioned as the hometown of longtime acquaintance of Putin, Kirill Shamalov)
* Biarritz, France (mentioned as the location of a villa owned by

100%|██████████| 5/5 [09:40<00:00, 116.14s/it]

(1, Boston, Massachusetts, Endocrinology Research Center, Moscow State University, 2, Boston, the Endocrinology Research Center, Maria Vladimirovna Vorontsova, Boston, 3, Endocrinology Research Center, Moscow, Maria Vladimirovna Vorontsova, Moscow State University, Katerina Vladimirovna Tikhonova, the National Intellectual Reserve Center, St. Petersburg, Putin, Kirill Shamalov, Biarritz, France, Katerina Vladimirovna Tikhonova)
Time taken: 00:03:27


In [40]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None


In [41]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Extract locations from the most specific pass

In [42]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [43]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

100%|██████████| 5/5 [00:00<?, ?it/s]


In [44]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None,None


In [45]:
df

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None,None


## Get the coordinates

In [47]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [48]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [49]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

100%|██████████| 5/5 [00:00<00:00,  8.76it/s]


In [50]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library,"[-71.0342146, 42.316274]"
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue,"[-71.0533982, 42.3527569]"
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU,"[-71.1053991, 42.3504997]"
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium,"[-71.2643465, 42.0909458]"
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None,None,None


## Geocode locations

In [51]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [52]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"]
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [53]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

100%|██████████| 5/5 [00:01<00:00,  3.84it/s]


,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library,"[-71.0342146, 42.316274]",090901,025
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue,"[-71.0533982, 42.3527569]",070104,025
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU,"[-71.1053991, 42.3504997]",010103,025
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium,"[-71.2643465, 42.0909458]",410100,021
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None,None,None,None,None


In [54]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

2


Explicit_Pass
JFK Library    1
BU             1
Name: count, dtype: int64

In [55]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

0


Series([], Name: count, dtype: int64)

In [56]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

2


NER_Prediction
600 Atlantic Avenue    1
Gillette Stadium       1
Name: count, dtype: int64

In [58]:
df.head(10)

,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library,"[-71.0342146, 42.316274]",090901,025
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue,"[-71.0533982, 42.3527569]",070104,025
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU,"[-71.1053991, 42.3504997]",010103,025
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium,"[-71.2643465, 42.0909458]",410100,021
8377,00000180-0377-d400-a7d5-17ffa6080000,Putin daughters were just sanctioned. Here wha...,The U.S. has announced additional sanctions ag...,None,None,None,None,None,None,None


In [59]:
len(df)

5

In [60]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [61]:
print(len(df))
df.head(10)

4


,_id,hl1,body,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
7670,0000017f-17c4-d83b-a97f-77d4c6750001,Due to COVID JFK Library celebrates Presidents...,The John F. Kennedy Presidential Library and M...,JFK Library,None,None,JFK Library,"[-71.0342146, 42.316274]",090901,025
8714,00000180-895d-d69c-ade1-b95d5f6c0001,things to know as the Fed embarks on its bigge...,The Federal Reserve is about to deliver its bi...,None,None,600 Atlantic Avenue,600 Atlantic Avenue,"[-71.0533982, 42.3527569]",070104,025
12349,00000186-8013-d717-adce-8a1f9b140002,He watched the Koons balloon dog fall and shat...,Welcome to new NPR series where we spotlight t...,BU,None,None,BU,"[-71.1053991, 42.3504997]",010103,025
2237,00000177-98e7-d244-a57f-fbffb7ec0001,Online Offers To Escort Seniors To Vaccine App...,People are posting online solicitations to giv...,None,None,Gillette Stadium,Gillette Stadium,"[-71.2643465, 42.0909458]",410100,021


## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns